# GWAS Tutorial by sgkit

This notebook is an sgkit port of Hail’s GWAS Tutorial, which demonstrates how to run a genome-wide SNP association test. Readers are encouraged to read the Hail tutorial alongside this one for more background, and to see the motivation behind some of the steps.

Note that some of the results do not exactly match the output from Hail. Also, since sgkit is still a 0.x release, its API is still subject to non-backwards compatible changes.

In [2]:
import sgkit as sg
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path
import requests

xr.set_options(display_expand_attrs=False, display_expand_data_vars=True);


In [3]:
## Importing data from vcf

vcf_path = "/home/sukui/01.data/03.raw_data/1kg/ALL.chr22.vcz"
pheno_file = "/home/sukui/01.data/03.raw_data/1kg/integrated_call_samples_v3.20250704.ALL.ped"

ds = sg.load_dataset(vcf_path)
df_pheno = pd.read_csv(pheno_file, sep='\t', index_col="Individual ID")

In [4]:
ds

<xarray.Dataset> Size: 14GB
Dimensions:                (variants: 1103547, samples: 2504, ploidy: 2,
                            contigs: 86, filters: 1, region_index_values: 1104,
                            region_index_fields: 6, alt_alleles: 8,
                            INFO_CIEND_dim: 2, INFO_CIPOS_dim: 2,
                            INFO_MC_dim: 4, INFO_MEINFO_dim: 4, INFO_VT_dim: 2,
                            alleles: 9)
Dimensions without coordinates: variants, samples, ploidy, contigs, filters,
                                region_index_values, region_index_fields,
                                alt_alleles, INFO_CIEND_dim, INFO_CIPOS_dim,
                                INFO_MC_dim, INFO_MEINFO_dim, INFO_VT_dim,
                                alleles
Data variables: (12/44)
    call_genotype          (variants, samples, ploidy) int8 6GB dask.array<chunksize=(1000, 2504, 2), meta=np.ndarray>
    call_genotype_mask     (variants, samples, ploidy) bool 6GB dask.array<chunksize=(1000, 2504, 2), meta=np.ndarray>
    call_genotype_phased   (variants, samples) bool 3GB dask.array<chunksize=(1000, 2504), meta=np.ndarray>
    contig_id              (contigs) object 688B dask.array<chunksize=(86,), meta=np.ndarray>
    contig_length          (contigs) float64 688B dask.array<chunksize=(86,), meta=np.ndarray>
    filter_description     (filters) object 8B dask.array<chunksize=(1,), meta=np.ndarray>
    ...                     ...
    variant_filter         (variants, filters) bool 1MB dask.array<chunksize=(1000, 1), meta=np.ndarray>
    variant_id             (variants) object 9MB dask.array<chunksize=(1000,), meta=np.ndarray>
    variant_id_mask        (variants) bool 1MB dask.array<chunksize=(1000,), meta=np.ndarray>
    variant_length         (variants) int32 4MB dask.array<chunksize=(1000,), meta=np.ndarray>
    variant_position       (variants) int32 4MB dask.array<chunksize=(1000,), meta=np.ndarray>
    variant_quality        (variants) float32 4MB dask.array<chunksize=(1000,), meta=np.ndarray>
Attributes: (3)

In [ ]:
# To join the annotation data with the genetic data, we convert it to Xarray, then do a join.
ds_annotations = pd.DataFrame.to_xarray(df_pheno).rename({"Individual ID":"samples"})
# 3. 如果 ds 中的 samples 已经是坐标，先重置
ds_aligned = ds.set_index({"samples": "sample_id"})
ds_aligned = ds_aligned.merge(ds_annotations, join="left")
ds_aligned.reset_index("samples").reset_coords(drop=True)
ds_aligned


<xarray.Dataset> Size: 14GB
Dimensions:                (samples: 2504, variants: 1103547, ploidy: 2,
                            contigs: 86, filters: 1, region_index_values: 1104,
                            region_index_fields: 6, alt_alleles: 8,
                            INFO_CIEND_dim: 2, INFO_CIPOS_dim: 2,
                            INFO_MC_dim: 4, INFO_MEINFO_dim: 4, INFO_VT_dim: 2,
                            alleles: 9)
Coordinates:
  * samples                (samples) object 20kB 'HG00096' ... 'NA21144'
Dimensions without coordinates: variants, ploidy, contigs, filters,
                                region_index_values, region_index_fields,
                                alt_alleles, INFO_CIEND_dim, INFO_CIPOS_dim,
                                INFO_MC_dim, INFO_MEINFO_dim, INFO_VT_dim,
                                alleles
Data variables: (12/59)
    call_genotype          (variants, samples, ploidy) int8 6GB dask.array<chunksize=(1000, 2504, 2), meta=np.ndarray>
    call_genotype_mask     (variants, samples, ploidy) bool 6GB dask.array<chunksize=(1000, 2504, 2), meta=np.ndarray>
    call_genotype_phased   (variants, samples) bool 3GB dask.array<chunksize=(1000, 2504), meta=np.ndarray>
    contig_id              (contigs) object 688B dask.array<chunksize=(86,), meta=np.ndarray>
    contig_length          (contigs) float64 688B dask.array<chunksize=(86,), meta=np.ndarray>
    filter_description     (filters) object 8B dask.array<chunksize=(1,), meta=np.ndarray>
    ...                     ...
    Children               (samples) object 20kB '0' '0' '0' '0' ... '0' '0' '0'
    Other Comments         (samples) object 20kB '0' '0' '0' '0' ... '0' '0' '0'
    phase 3 genotypes      (samples) int64 20kB 1 1 1 1 1 1 1 ... 1 1 1 1 1 1 1
    related genotypes      (samples) int64 20kB 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0
    omni genotypes         (samples) int64 20kB 1 1 1 1 1 1 1 ... 1 1 1 1 1 1 1
    affy_genotypes         (samples) int64 20kB 1 1 1 1 1 1 0 ... 1 1 1 1 1 1 1
Attributes: (3)

In [7]:
# get superpolulation
super_pop_mapping = {
            # 非洲 (AFR)
            "YRI": 0,  # "AFR",
            "LWK": 0,  # "AFR",
            "GWD": 0,  # "AFR",
            "MSL": 0,  # "AFR",
            "ESN": 0,  # "AFR",
            "ACB": 0,  # "AFR",
            # 欧洲 (EUR)
            "GBR": 1,  # "EUR",
            "FIN": 1,  # "EUR",
            "IBS": 1,  # "EUR",
            "CEU": 1,  # "EUR",
            "TSI": 1,  # "EUR",
            # 东亚 (EAS)
            "CHS": 2,  # "EAS",
            "CHB": 2,  # "EAS",
            "JPT": 2,  # "EAS",
            "CDX": 2,  # "EAS",
            "CHD": 2,  # "EAS",
            "KHV": 2,  # "EAS",
            # 南亚 (SAS)
            "GIH": 3,  # "SAS",
            "PJL": 3,  # "SAS",
            "BEB": 3,  # "SAS",
            "ITU": 3,  # "SAS",
            "STU": 3,  # "SAS",
            # 美洲 (AMR)
            "MXL": 4,  # "AMR",
            "PUR": 4,  # "AMR",
            "CLM": 4,  # "AMR",
            "PEL": 4,  # "AMR",
            # 混合群体 - 根据遗传背景分类
            "ASW": 5,  # "UNK",  # 非裔美国人，归为AFR
        }


df_pheno["SuperPopulation"] = df_pheno["Population"].map(super_pop_mapping)
df_pheno.head()

,Family ID,Paternal ID,Maternal ID,Gender,Phenotype,Population,Relationship,Siblings,Second Order,Third Order,Children,Other Comments,phase 3 genotypes,related genotypes,omni genotypes,affy_genotypes,SuperPopulation
Individual ID,,,,,,,,,,,,,,,,,
HG00096,HG00096,0,0,1,0,GBR,unrel,0,0,0,0,0,1,0,1,1,1
HG00097,HG00097,0,0,2,0,GBR,unrel,0,0,0,0,0,1,0,1,1,1
HG00098,HG00098,0,0,1,0,GBR,unrel,0,0,0,0,0,0,0,1,1,1
HG00099,HG00099,0,0,2,0,GBR,unrel,0,0,0,0,0,1,0,1,1,1
HG00100,HG00100,0,0,2,0,GBR,unrel,0,0,0,0,0,1,0,1,1,1


In [59]:
import sgkit as sg
import xarray as xr

# 1. 首先计算变异统计信息，包括等位基因计数
ds = sg.variant_stats(ds)

# 2. 检查等位基因数
print("=== 等位基因统计 ===")
print(f"总变异数: {len(ds.variants)}")

if 'variant_allele_count' in ds:
    # 查看等位基因分布
    allele_counts = ds.variant_allele_count.values
    unique_counts, counts = np.unique(allele_counts, return_counts=True)
    
    print("\n等位基因数分布:")
    for uc, cnt in zip(unique_counts, counts):
        print(f"  {uc}个等位基因: {cnt}个变异 ({cnt/len(ds.variants)*100:.2f}%)")
    
    # 3. 过滤掉非双等位基因（等位基因数 != 2）
    biallelic_mask = ds.variant_allele_count == 2
    ds_biallelic = ds.sel(variants=biallelic_mask)
    
    print(f"\n过滤后双等位基因变异数: {len(ds_biallelic.variants)}")
    
    # 4. 现在进行 HWE 检验
    ds_hwe = sg.hardy_weinberg_test(ds_biallelic)
    
    # 5. 应用过滤条件
    ds_filtered = ds_hwe.sel(variants=(
        (ds_hwe.variant_allele_frequency[:, 1] > 0.01) & 
        (ds_hwe.variant_hwe_p_value > 1e-6)
    ))
    
    print(f"\n最终结果:")
    print(f"Samples: {len(ds_filtered.samples)}")
    print(f"Variants: {len(ds_filtered.variants)}")
    
else:
    print("错误: 没有找到 variant_allele_count 变量")
    print("可用的变量:", list(ds.data_vars))

/home/sukui/.local/lib/python3.10/site-packages/sgkit/utils.py:222: MergeWarning: The following variables in the input dataset will be replaced in the output: variant_allele_frequency, variant_allele_total, variant_call_rate, variant_n_called, variant_n_het, variant_n_hom_alt, variant_n_hom_ref, variant_n_non_ref
  warnings.warn(


=== 等位基因统计 ===
总变异数: 1103547

等位基因数分布:
  0个等位基因: 7722214个变异 (699.76%)
  1个等位基因: 453987个变异 (41.14%)
  2个等位基因: 123731个变异 (11.21%)
  3个等位基因: 61378个变异 (5.56%)
  4个等位基因: 38691个变异 (3.51%)
  5个等位基因: 27454个变异 (2.49%)
  6个等位基因: 20933个变异 (1.90%)
  7个等位基因: 16445个变异 (1.49%)
  8个等位基因: 14128个变异 (1.28%)
  9个等位基因: 12403个变异 (1.12%)
  10个等位基因: 10688个变异 (0.97%)
  11个等位基因: 9298个变异 (0.84%)
  12个等位基因: 8335个变异 (0.76%)
  13个等位基因: 7216个变异 (0.65%)
  14个等位基因: 6584个变异 (0.60%)
  15个等位基因: 5937个变异 (0.54%)
  16个等位基因: 5877个变异 (0.53%)
  17个等位基因: 5269个变异 (0.48%)
  18个等位基因: 4847个变异 (0.44%)
  19个等位基因: 4573个变异 (0.41%)
  20个等位基因: 4410个变异 (0.40%)
  21个等位基因: 4260个变异 (0.39%)
  22个等位基因: 3793个变异 (0.34%)
  23个等位基因: 3483个变异 (0.32%)
  24个等位基因: 3378个变异 (0.31%)
  25个等位基因: 3122个变异 (0.28%)
  26个等位基因: 3075个变异 (0.28%)
  27个等位基因: 3121个变异 (0.28%)
  28个等位基因: 2795个变异 (0.25%)
  29个等位基因: 2883个变异 (0.26%)
  30个等位基因: 2446个变异 (0.22%)
  31个等位基因: 2439个变异 (0.22%)
  32个等位基因: 2347个变异 (0.21%)
  33个等位基因: 2324个变异 (0.21%)
  34个等位基因: 2148个变异 (0.19%)
  35个等位

IndexError: 2-dimensional boolean indexing is not supported. 

In [58]:
ac = ds.variant_allele_count.values
ac.shape

(1103547, 9)